In [1]:
import xml.etree.ElementTree as ET  # For parsing OME XML metadata
from tifffile import TiffFile       # For opening OME-TIFF files

In [2]:
_OME_NS = {"ome": "http://www.openmicroscopy.org/Schemas/OME/2016-06"}

def load_ome_physical_info(fpath):
    """Return (info: dict, pixels_to_physical: callable) and print a summary."""
    with TiffFile(fpath) as tif:
        ome_xml = tif.ome_metadata
        dtype = tif.pages[0].dtype if tif.pages else None

    root = ET.fromstring(ome_xml)

    # Core Pixels block
    px = root.find(".//ome:Image/ome:Pixels", _OME_NS)
    sizeX = int(px.get("SizeX"))
    sizeY = int(px.get("SizeY"))
    sizeZ = int(px.get("SizeZ", 1))
    sizeC = int(px.get("SizeC", 1))
    sizeT = int(px.get("SizeT", 1))
    dim_order = px.get("DimensionOrder")
    dtype_xml = px.get("Type")

    # Physical sizes (+ units if present; OME often omits => assume µm)
    sx = float(px.get("PhysicalSizeX"))
    sy = float(px.get("PhysicalSizeY"))
    sx_unit = px.get("PhysicalSizeXUnit") or "µm"
    sy_unit = px.get("PhysicalSizeYUnit") or "µm"

    # Optional global origin: Plate origin + WellSample position (if present)
    plate = root.find(".//ome:Plate", _OME_NS)
    well_sample = root.find(".//ome:Well/ome:WellSample", _OME_NS)

    well_origin_x = float(plate.get("WellOriginX")) if plate is not None and plate.get("WellOriginX") else 0.0
    well_origin_y = float(plate.get("WellOriginY")) if plate is not None and plate.get("WellOriginY") else 0.0
    pos_x = float(well_sample.get("PositionX")) if well_sample is not None and well_sample.get("PositionX") else 0.0
    pos_y = float(well_sample.get("PositionY")) if well_sample is not None and well_sample.get("PositionY") else 0.0

    origin_x = well_origin_x + pos_x
    origin_y = well_origin_y + pos_y
    origin_unit = "µm"  # OME positions default to µm unless specified otherwise

    # Channels (names)
    ch_names = [
        ch.get("Name") or f"Channel-{i}"
        for i, ch in enumerate(root.findall(".//ome:Image/ome:Pixels/ome:Channel", _OME_NS))
    ]

    # Acquisition date (optional)
    acq = root.find(".//ome:Image/ome:AcquisitionDate", _OME_NS)
    acq_time = acq.text if acq is not None else None

    info = {
        "size": {"X": sizeX, "Y": sizeY, "Z": sizeZ, "C": sizeC, "T": sizeT},
        "dimension_order": dim_order,
        "dtype_xml": dtype_xml,
        "dtype_tiff": str(dtype) if dtype is not None else None,
        "physical_pixel_size": {"X": sx, "X_unit": sx_unit, "Y": sy, "Y_unit": sy_unit},
        "origin": {"X": origin_x, "Y": origin_y, "unit": origin_unit,
                   "components": {"WellOriginX": well_origin_x, "WellOriginY": well_origin_y,
                                  "PositionX": pos_x, "PositionY": pos_y}},
        "channels": ch_names,
        "acquisition_time": acq_time,
    }

    # Pretty print
    print("=== OME Physical Metadata ===")
    print(f"Size (XYZCT): {sizeX} x {sizeY} x {sizeZ} x {sizeC} x {sizeT}  |  Order: {dim_order}")
    print(f"DType (OME/TIFF): {dtype_xml} / {info['dtype_tiff']}")
    print(f"Pixel size: X={sx} {sx_unit}  Y={sy} {sy_unit}")
    print(f"Origin (best-effort): X={origin_x:.3f} {origin_unit}, Y={origin_y:.3f} {origin_unit}")
    print(f"  Components: WellOrigin=({well_origin_x}, {well_origin_y}), "
          f"Position=({pos_x}, {pos_y}) [{origin_unit}]")
    print(f"Channels ({len(ch_names)}): {', '.join(ch_names) if ch_names else '—'}")
    print(f"Acquired: {acq_time}")
    print("=============================")

    # Mapper
    def pixels_to_physical(ix, iy):
        """Map pixel indices (ix, iy) -> physical coords (X, Y) in origin_unit."""
        return origin_x + ix * sx, origin_y + iy * sy

    return info, pixels_to_physical


# Test image Day 5
fpath = "/scratch/indikar_root/indikar1/jrcwycy/HYB/HYB04/images_day5/test-scene10-wellC6_s10.ome.tiff"
info, p2p = load_ome_physical_info(fpath)

# Example: top-left (0,0) and center pixel
x0, y0 = p2p(0, 0)
xc, yc = p2p(info["size"]["X"] // 2, info["size"]["Y"] // 2)
print(f"Example coords: (0,0) -> ({x0:.3f}, {y0:.3f}) {info['origin']['unit']}")
print(f"Center pixel -> ({xc:.3f}, {yc:.3f}) {info['origin']['unit']}")

=== OME Physical Metadata ===
Size (XYZCT): 498 x 367 x 1 x 9 x 1  |  Order: XYZTC
DType (OME/TIFF): uint8 / uint8
Pixel size: X=2.724 µm  Y=2.724 µm
Origin (best-effort): X=49428.137 µm, Y=26654.152 µm
  Components: WellOrigin=(-9900.0, -2750.0), Position=(59328.1367, 29404.1523) [µm]
Channels (3): H3342, AF488, AF594
Acquired: 2026-03-04T18:04:55.1451862Z
Example coords: (0,0) -> (49428.137, 26654.152) µm
Center pixel -> (50106.413, 27152.644) µm


In [3]:
# Test image Day 8
fpath = "/scratch/indikar_root/indikar1/jrcwycy/HYB/HYB04/images_day8/test-scene11-wellD2_s11.ome.tiff"
info, p2p = load_ome_physical_info(fpath)

# Example: top-left (0,0) and center pixel
x0, y0 = p2p(0, 0)
xc, yc = p2p(info["size"]["X"] // 2, info["size"]["Y"] // 2)
print(f"Example coords: (0,0) -> ({x0:.3f}, {y0:.3f}) {info['origin']['unit']}")
print(f"Center pixel -> ({xc:.3f}, {yc:.3f}) {info['origin']['unit']}")

=== OME Physical Metadata ===
Size (XYZCT): 498 x 367 x 1 x 9 x 1  |  Order: XYZTC
DType (OME/TIFF): uint8 / uint8
Pixel size: X=2.724 µm  Y=2.724 µm
Origin (best-effort): X=13428.137 µm, Y=35654.152 µm
  Components: WellOrigin=(-9900.0, -2750.0), Position=(23328.1367, 38404.1523) [µm]
Channels (3): H3342, AF488, AF594
Acquired: 2026-03-10T17:05:20.3777664Z
Example coords: (0,0) -> (13428.137, 35654.152) µm
Center pixel -> (14106.413, 36152.644) µm
